# LLM Inference Benchmark Suite — Colab Orchestrator

Compares FP16, GPTQ, AWQ, GGUF, and TensorRT-LLM inference on a single
open-weight model, using **one isolated virtual environment per technique**
(see `README.md` and `docs/environment_notes.md` for why).

**Recommended runtime: A100 (40GB).** Phases 1-4 also work on a T4;
Phase 5 (TensorRT-LLM) requires Ampere or newer and will fail fast with a
clear assertion error otherwise.


In [ ]:
# Phase 0 -- Clone repo and confirm GPU
!git clone -q https://github.com/arkanathroy/llm-inference-benchmark-suite.git 2>/dev/null || echo "repo already present"
%cd llm-inference-benchmark-suite

!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv


In [ ]:
# Phase 0.5 -- Shared Hugging Face cache (avoid re-downloading Qwen per env)
import os
from pathlib import Path

HF_CACHE_DIR = "/content/hf_cache"
Path(HF_CACHE_DIR).mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

from google.colab import userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab secrets.")
except Exception:
    print("No HF_TOKEN secret found -- proceeding unauthenticated (fine for public models like Qwen2.5-3B-Instruct).")

print(f"HF cache set to {HF_CACHE_DIR} -- shared across all envs/*/venv subprocesses "
      f"since HF_HOME is inherited via os.environ by every run_in_env() subprocess.")

In [ ]:
# Phase 0.6 -- Imports shared by every technique cell below
import sys, subprocess, time
sys.path.insert(0, "src")
from config import CONFIG
from env_runner import run_in_env
from server_utils import wait_for_health, stop_server, start_vllm_server, start_llamacpp_server
from pathlib import Path


def teardown_env(env_name):
    """Delete a technique's venv to reclaim disk space once its benchmark is done."""
    venv_path = Path(f"envs/{env_name}/venv")
    if venv_path.exists():
        get_ipython().system(f"rm -rf {venv_path}")
        print(f"Deleted {venv_path} to reclaim disk space.")
    get_ipython().system("df -h /content | tail -1")


def run_accuracy_eval(env_name, base_url, model_id, technique_name,
                      limit=CONFIG.eval.limit, num_concurrent=1):
    """
    Run lm-eval accuracy tasks against a server that is already live,
    right after its throughput benchmark, avoiding a second server
    start/stop cycle.

    WHAT changed: added limit and num_concurrent as explicit optional
    parameters. Previously limit came only from CONFIG.eval.limit (fixed
    at whatever the config specifies, e.g. 200) and num_concurrent was
    hardcoded to 1 inside the model_args f-string. Both are now
    overridable per call, defaulting to limit=None (full dataset) and
    num_concurrent=1 (unchanged/original throughput) so existing
    behavior only changes if the caller opts in.

    WHY keep CONFIG.eval.limit as the smoke-test value rather than
    hardcoding 200 here: CONFIG.eval.limit is still the single source
    of truth for what "smoke test" means -- if that value ever changes
    in config.py, both the config-driven smoke-test call AND this
    function's fallback stay in sync automatically. This function's
    own limit=None default is deliberately the OPPOSITE of
    CONFIG.eval.limit (full dataset, not the smoke-test slice) --
    calling run_accuracy_eval() with zero extra args now runs the real
    evaluation, while calling it with limit=CONFIG.eval.limit
    explicitly reproduces the fast smoke test. This is the inverse of
    the previous behavior, where the config value WAS the only
    behavior available -- see the notebook cell below for how both
    modes get called out explicitly, in the intended order
    (smoke test first, full run second).

    EFFECT of limit=None vs limit=CONFIG.eval.limit (e.g. 200):
    - limit=200 (smoke test): ~90s per technique on hellaswag+arc_easy,
      confirms env -> server -> lm-eval -> results pipeline works
      before committing to a long run. Stderr on acc/acc_norm is wide
      (~+/-0.03), unsuitable for citing real accuracy deltas.
    - limit=None (full dataset): full hellaswag (10,042 examples) +
      arc_easy (2,376 examples) -- stderr tightens to roughly +/-0.01
      or better, the resolution actually needed to distinguish FP16 vs
      GPTQ vs AWQ accuracy deltas that are themselves often only
      0.5-2 accuracy points apart. lm-eval's own runtime warning
      ("--limit SHOULD ONLY BE USED FOR TESTING") is the direct
      justification for treating limit=None as the reportable run.

    EFFECT of num_concurrent=1 vs num_concurrent=4+:
    - num_concurrent=1 serializes every lm-eval request against the
      vLLM server one at a time. Prior full run logs showed a flat
      ~16-18 it/s ceiling that looks latency-bound (HTTP round-trip +
      tokenization), not GPU-compute-bound.
    - num_concurrent=4 lets lm-eval keep 4 loglikelihood requests
      in flight, giving vLLM's continuous batching scheduler
      something to actually batch -- expect roughly 2-3x wall-clock
      improvement in practice (not a clean 4x, since Phase A's
      benchmark showed ~99.9% GPU util even at batch_size=1, so
      decode compute is already partially saturated).
    - Risk: kept as an explicit opt-in argument rather than a new
      default, since lm-eval's default of 1 is the safest choice for
      the smoke test call -- num_concurrent=4 is only used on the
      full run further below.
    """
    tasks_arg = ",".join(CONFIG.eval.tasks)

    print(f"[accuracy_eval] {technique_name}: running lm-eval tasks={tasks_arg} "
          f"limit={limit} num_concurrent={num_concurrent} against {base_url} "
          f"(this can take a few minutes)...")

    output_path = f"results/lm_eval_{technique_name}"
    model_args = (
        f"base_url={base_url}/completions,model={model_id},"
        f"num_concurrent={num_concurrent},max_retries=3,tokenized_requests=False"
    )

    cmd_args = [
        "lm_eval", "--model", "local-completions",
        "--model_args", model_args,
        "--tasks", tasks_arg,
        "--output_path", output_path,
    ]
    # WHAT: only appends --limit when a limit is actually in effect.
    # WHY: omitting the flag entirely for the full run (rather than
    # passing a very large number) matches lm-eval's own documented
    # "full dataset, no shortcuts" behavior and avoids the internal
    # testing-mode warning being logged on a run meant to be final.
    if limit is not None:
        cmd_args += ["--limit", str(limit)]

    run_in_env(env_name, "-m", cmd_args)
    print(f"[accuracy_eval] {technique_name}: complete, results in {output_path}")

In [ ]:
# Phase 0.7 -- Start Loki + Promtail + Grafana for live log observability
# See docs/observability.md for the full stepwise guide (dashboards, panels,
# how to read logs live while a phase is running).
!bash observability/setup_observability.sh

# Expose Grafana (port 3000) via a public URL so you can view dashboards
# from your browser while Colab keeps running.
from google.colab.output import eval_js
grafana_url = eval_js("google.colab.kernel.proxyPort(3000)")
print(f"Grafana dashboard: {grafana_url}")
print("Open this URL, add a Loki data source at http://localhost:3100, "
      "then follow docs/observability.md to build the live progress panel.")

In [ ]:
# Phase A -- FP16 baseline: build env, benchmark, teardown
!bash envs/fp16/setup.sh


model_id = CONFIG.model.hf_repo
port = CONFIG.server.vllm_port
base_url = f"http://{CONFIG.server.host}:{port}"


server_proc = start_vllm_server(
    "envs/fp16/venv/bin/python", model_id, CONFIG.server.host, port,
    CONFIG.server.gpu_memory_utilization, CONFIG.model.max_model_len,
)
wait_for_health(base_url, server_proc)


run_in_env("fp16", "src/benchmark_runner.py",
           ["--technique", "fp16", "--base_url", f"{base_url}/v1", "--model_id", model_id])


# WHAT: two run_accuracy_eval calls -- fast smoke test, then full run.
# WHY: reuses CONFIG.eval.limit (e.g. 200) explicitly for the smoke
# test so its meaning stays tied to config, then a second call with
# limit=None, num_concurrent=4 for the real, citable accuracy numbers.
# Writing to two different technique_name suffixes keeps both outputs
# on disk simultaneously under results/ for inspection.
# EFFECT: smoke test costs ~90s and gates the expensive run. Only
# results/lm_eval_fp16_full should be used in the final FP16 vs GPTQ
# vs AWQ comparison table -- results/lm_eval_fp16 is diagnostic only.
run_accuracy_eval("fp16", f"{base_url}/v1", model_id, "fp16")

run_accuracy_eval("fp16", f"{base_url}/v1", model_id, "fp16_full",
                   limit=None, num_concurrent=4)


stop_server(server_proc)
teardown_env("fp16")

In [ ]:
# Phase B -- GPTQ: build env, quantize, benchmark, teardown
!bash envs/gptq/setup.sh

run_in_env("gptq", "src/quantize_gptq.py", [])

gptq_model_dir = CONFIG.gptq.output_dir
port = CONFIG.server.vllm_port
base_url = f"http://{CONFIG.server.host}:{port}"

server_proc = start_vllm_server(
    "envs/gptq/venv/bin/python", gptq_model_dir, CONFIG.server.host, port,
    CONFIG.server.gpu_memory_utilization, CONFIG.model.max_model_len, quantization="gptq",
)
wait_for_health(base_url, server_proc)

run_in_env("gptq", "src/benchmark_runner.py",
           ["--technique", "gptq", "--base_url", f"{base_url}/v1", "--model_id", gptq_model_dir])


# WHAT: two run_accuracy_eval calls -- fast smoke test, then full run.
# WHY gptq_model_dir (not a HF repo string) is passed as the model
# identifier: the GPTQ server was started against the LOCAL quantized
# checkpoint written by quantize_gptq.py (CONFIG.gptq.output_dir), not
# against the original HF repo -- vLLM's internal model registry keys
# off whatever path/id start_vllm_server loaded, so model_args must
# echo that same identifier for lm-eval's requests to route correctly.
# This mirrors --model_id gptq_model_dir already used in the
# benchmark_runner.py call two lines above.
# EFFECT: results/lm_eval_gptq_full is what belongs in the FP16 vs
# GPTQ vs AWQ comparison table -- results/lm_eval_gptq is diagnostic
# smoke-test output only.
run_accuracy_eval("gptq", f"{base_url}/v1", gptq_model_dir, "gptq")

run_accuracy_eval("gptq", f"{base_url}/v1", gptq_model_dir, "gptq_full",
                   limit=None, num_concurrent=4)

stop_server(server_proc)
teardown_env("gptq")
# Optionally also free the quantized weights once benchmarked:
# get_ipython().system(f"rm -rf {gptq_model_dir}")

In [ ]:
# Phase C -- AWQ: build env, quantize, benchmark, teardown
!bash envs/awq/setup.sh

run_in_env("awq", "src/quantize_awq.py", [])

awq_model_dir = CONFIG.awq.output_dir
port = CONFIG.server.vllm_port
base_url = f"http://{CONFIG.server.host}:{port}"

server_proc = start_vllm_server(
    "envs/awq/venv/bin/python", awq_model_dir, CONFIG.server.host, port,
    CONFIG.server.gpu_memory_utilization, CONFIG.model.max_model_len, quantization="awq",
)
wait_for_health(base_url, server_proc)

run_in_env("awq", "src/benchmark_runner.py",
           ["--technique", "awq", "--base_url", f"{base_url}/v1", "--model_id", awq_model_dir])


# WHAT: two run_accuracy_eval calls -- fast smoke test, then full run.
# WHY awq_model_dir is passed: identical reasoning to GPTQ above --
# the AWQ server loaded the local quantized checkpoint at
# CONFIG.awq.output_dir, so that path is the model identifier the
# live server actually recognizes, matching the --model_id argument
# already used for benchmark_runner.py.
# EFFECT: results/lm_eval_awq_full is the citable comparison-table
# source; results/lm_eval_awq is diagnostic only.
run_accuracy_eval("awq", f"{base_url}/v1", awq_model_dir, "awq")

run_accuracy_eval("awq", f"{base_url}/v1", awq_model_dir, "awq_full",
                   limit=None, num_concurrent=4)

stop_server(server_proc)
teardown_env("awq")
# get_ipython().system(f"rm -rf {awq_model_dir}")

In [ ]:
# Phase D -- GGUF: build env, convert+quantize, benchmark, teardown
!bash envs/gguf/setup.sh

run_in_env("gguf", "src/convert_gguf.py", [])

gguf_dir = CONFIG.gguf.output_dir
gguf_file = f"{gguf_dir}/model-Q4_K_M.gguf"
llamacpp_port = CONFIG.server.llamacpp_port
llamacpp_base_url = f"http://{CONFIG.server.host}:{llamacpp_port}"

server_proc = start_llamacpp_server(
    "envs/gguf/llama.cpp/build/bin/llama-server", gguf_file,
    CONFIG.server.host, llamacpp_port, CONFIG.model.max_model_len,
)
wait_for_health(llamacpp_base_url, server_proc)

run_in_env("gguf", "src/benchmark_runner.py",
           ["--technique", "gguf_q4_k_m", "--base_url", f"{llamacpp_base_url}/v1", "--model_id", gguf_file])


# WHAT: two run_accuracy_eval calls -- fast smoke test, then full run.
# WHY gguf_file (a specific .gguf file path, not even a directory) is
# passed: llama.cpp's server loads one concrete quantized GGUF file
# (model-Q4_K_M.gguf here), so that exact filepath is the identifier
# the llama-server process holds -- same value already passed as
# --model_id to benchmark_runner.py two lines above.
# WHY llamacpp_base_url instead of base_url: GGUF is served by
# llama-server on CONFIG.server.llamacpp_port, a completely separate
# process/port from the vLLM server used in fp16/gptq/awq -- reusing
# the vLLM base_url variable here would silently point lm-eval at the
# wrong (or dead) server.
# EFFECT: results/lm_eval_gguf_q4_k_m_full is the citable comparison-
# table source; results/lm_eval_gguf_q4_k_m is diagnostic only.
run_accuracy_eval("gguf", f"{llamacpp_base_url}/v1", gguf_file, "gguf_q4_k_m")

run_accuracy_eval("gguf", f"{llamacpp_base_url}/v1", gguf_file, "gguf_q4_k_m_full",
                   limit=None, num_concurrent=4)

stop_server(server_proc)
teardown_env("gguf")
# get_ipython().system(f"rm -rf {gguf_dir}")

In [ ]:
# Phase E -- TensorRT-LLM (Ampere+/A100 only): build env, build engine, benchmark, teardown
import torch
major, minor = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
if major >= 8:
    !bash envs/trtllm/setup.sh
    run_in_env("trtllm", "src/build_trtllm_engine.py", [])

    engine_dir = f"{CONFIG.trtllm.output_dir}/trt_engine"
    run_in_env("trtllm", "src/trtllm_bench.py",
               ["--engine_dir", engine_dir, "--output", "results/trtllm_benchmark.json"])

    teardown_env("trtllm")
    # get_ipython().system(f"rm -rf {CONFIG.trtllm.output_dir}")
else:
    print(f"Skipping TensorRT-LLM: GPU compute capability sm_{major}{minor} < sm_80 (needs Ampere+/A100).")

## Phase 9 -- Aggregate results and plot comparison charts

In [ ]:
# Phase 9.1 -- Load combined results CSV
import pandas as pd

results_csv = Path(CONFIG.output.results_dir) / CONFIG.output.csv_name
df = pd.read_csv(results_csv) if results_csv.exists() else pd.DataFrame()
df


In [ ]:
# Phase 9.2 -- Fold in TensorRT-LLM results (separate JSON schema) if present
import json

trtllm_json = Path("results/trtllm_benchmark.json")
if trtllm_json.exists():
    trt_results = json.load(open(trtllm_json))
    trt_df = pd.DataFrame(trt_results).rename(columns={
        "avg_latency_s": "avg_latency_s",
        "tokens_per_second": "avg_tokens_per_second",
    })
    df = pd.concat([df, trt_df], ignore_index=True, sort=False)

df


In [ ]:
# Phase 9.3 -- Plot tokens/sec by technique and batch size
import plotly.express as px

if not df.empty:
    fig = px.bar(
        df, x="batch_size", y="avg_tokens_per_second", color="technique",
        barmode="group", title="Throughput by technique and batch size",
        labels={"avg_tokens_per_second": "tokens / second", "batch_size": "batch size"},
    )
    fig.write_image("results/benchmark_comparison.png", scale=2)
    fig.show()
else:
    print("No results yet -- run Phases 3-7 first.")
